In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
import warnings

warnings.filterwarnings('ignore')

In [3]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')
original = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')

In [4]:
train['Churn'] = train['Churn'].map({'Yes': 1, 'No': 0}).astype(int)
original['Churn'] = original['Churn'].map({'Yes': 1, 'No': 0}).astype(int)

original['TotalCharges'] = pd.to_numeric(original['TotalCharges'], errors='coerce')
original['TotalCharges'].fillna(original['TotalCharges'].median(), inplace=True)
# original['TotalCharges'].fillna(original['MonthlyCharges'] * original['tenure'], inplace=True)

In [5]:
original.drop('customerID', axis=1, inplace=True)
train.drop('id', axis=1, inplace=True)
test.drop('id', axis=1, inplace=True)

In [6]:
cats = [
    'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService',
    'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup',
    'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies',
    'Contract', 'PaperlessBilling', 'PaymentMethod'
]

nums = ['tenure', 'MonthlyCharges', 'TotalCharges']

target = 'Churn'

new_nums = []
nums_as_cat = []

In [7]:
for col in nums:
    freq = pd.concat([train[col], original[col], test[col]]).value_counts(normalize=True)
    for df in [train, test, original]:
        df[f'Freq{col}'] = df[col].map(freq).fillna(0).astype('float32')
    new_nums.append(f'Freq{col}')

In [8]:
for col in cats:

    churn_mean = original.groupby(col)[target].mean()
    churn_std = original.groupby(col)[target].std()

    train[f"{col}ChurnMean"] = train[col].map(churn_mean)
    test[f"{col}ChurnMean"] = test[col].map(churn_mean)

    train[f"{col}ChurnStd"] = train[col].map(churn_std)
    test[f"{col}ChurnStd"] = test[col].map(churn_std)

In [9]:
q1 = train["MonthlyCharges"].quantile(0.25)
q3 = train["MonthlyCharges"].quantile(0.75)

for df in [train, test]:

    df["ChargeOutlier"] = (
        (df["MonthlyCharges"] < q1) |
        (df["MonthlyCharges"] > q3)
    ).astype(int)

    df['MCLog'] = np.log1p(df['MonthlyCharges'])
    df['TCLog'] = np.log1p(df['TotalCharges'])

    df['MCSqueezed'] = 1 / (df['MonthlyCharges'] + 1)
    df['TCSqueezed'] = 1 / (df['TotalCharges'] + 1)

    df["TenureLog"] = np.log1p(df["tenure"])
    df['TenureSqueezed'] = 1 / (df['tenure'] + 1)

    df["AvgMonthlySpend"] = df["TotalCharges"] / (df["tenure"] + 1)
    df["SpendDeviation"] = df["MonthlyCharges"] - df["AvgMonthlySpend"]
    df["MonthlyToTotalRatio"] = df["MonthlyCharges"] / (df["TotalCharges"] + 1)

    df["IsMonthToMonth"] = (df["Contract"] == "Month-to-month").astype(int)

    df["IsElectronicCheck"] = (df["PaymentMethod"] == "Electronic check").astype(int)
    df["AutoPayment"] = df["PaymentMethod"].str.contains("automatic").astype(int)

    df["ServiceCount"] = (
        (df["OnlineSecurity"] == "Yes").astype(int) +
        (df["OnlineBackup"] == "Yes").astype(int) +
        (df["DeviceProtection"] == "Yes").astype(int) +
        (df["TechSupport"] == "Yes").astype(int) +
        (df["StreamingTV"] == "Yes").astype(int) +
        (df["StreamingMovies"] == "Yes").astype(int)
    )

    df["IsFiber"] = (df["InternetService"] == "Fiber optic").astype(int)

    df["FiberHighCharge"] = (
        (df["InternetService"] == "Fiber optic") &
        (df["MonthlyCharges"] > df["MonthlyCharges"].median())
    ).astype(int)

    df["FiberNoSupport"] = (
        (df["InternetService"] == "Fiber optic") &
        (df["TechSupport"] == "No")
    ).astype(int)

    df["EarlyTenure"] = (df["tenure"] < 12).astype(int)
    df["HighChargeEarly"] = (
        (df["tenure"] < 12) &
        (df["MonthlyCharges"] > df["MonthlyCharges"].median())
    ).astype(int)

    df["TenureBucket"] = pd.cut(
        df["tenure"],
        bins=[0, 6, 12, 24, 48, 72],
        labels=False
    )

    df["VeryEarly"] = (df["tenure"] < 6).astype(int)

    df["LifetimeValue"] = df["MonthlyCharges"] * df["tenure"]

    df["ChargeRank"] = df["MonthlyCharges"].rank(pct=True)
    df["TenureRank"] = df["tenure"].rank(pct=True)

new_nums = [col for col in df.columns.tolist() if col not in (nums + cats + [target])]

In [10]:
for col in cats + nums:
    tmp = original.groupby(col)[target].mean()
    name = f"OrigP{col}"
    train = train.merge(tmp.rename(name), on=col, how="left")
    test = test.merge(tmp.rename(name), on=col, how="left")
    for df in [train, test]:
        df[name] = df[name].fillna(0.5).astype('float32')
    new_nums.append(name)

In [11]:
X, y = train.drop(target, axis=1), train[target]

In [12]:
from sklearn.preprocessing import OneHotEncoder, LabelEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

label_cols = [
    'gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
    'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
    'TechSupport', 'StreamingTV', 'StreamingMovies', 'PaperlessBilling'
]

onehot_cols = ['PaymentMethod']

ord_mapping = {
    'Contract': ['Two year', 'One year', 'Month-to-month'],
    'MultipleLines': ['No phone service', 'No', 'Yes'],
    'InternetService': ['No', 'DSL', 'Fiber optic'],
    'OnlineSecurity': ['No internet service', 'No', 'Yes'],
    'OnlineBackup': ['No internet service', 'No', 'Yes'],
    'DeviceProtection': ['No internet service', 'No', 'Yes'],
    'TechSupport': ['No internet service', 'No', 'Yes'],
    'StreamingTV': ['No internet service', 'No', 'Yes'],
    'StreamingMovies': ['No internet service', 'No', 'Yes'],
    'Partner': ['No', 'Yes'],
    'Dependents': ['No', 'Yes'],
    'PhoneService': ['No', 'Yes'],
    'PaperlessBilling': ['No', 'Yes'],
    'gender': ['Male', 'Female']
}

ord_cols = list(ord_mapping.keys())

preprocessor = ColumnTransformer(
    transformers=[
        ('onehot', OneHotEncoder(drop='first', sparse_output=False), onehot_cols),
        ('ordinal', OrdinalEncoder(categories=[ord_mapping[col] for col in ord_cols]), ord_cols)
    ],
    remainder='passthrough'
)

pipeline = Pipeline([
    ('preprocessor', preprocessor)
])

pipeline.set_output(transform="pandas")

X_train_encoded = pipeline.fit_transform(X)
X_test_encoded = pipeline.transform(test)

In [15]:
import json

with open('xgboost-params.json', 'r') as file:
    params = json.load(file)

In [18]:
params

{'learning_rate': 0.022200439192185714,
 'max_depth': 4,
 'min_child_weight': 7,
 'subsample': 0.8852275843032212,
 'colsample_bytree': 0.5048287064762882,
 'gamma': 0.6532478645748768,
 'reg_alpha': 5.190282638441493,
 'reg_lambda': 2.7585060147705223}

In [20]:
X_train_encoded

,onehot__PaymentMethod_Credit card (automatic),onehot__PaymentMethod_Electronic check,onehot__PaymentMethod_Mailed check,ordinal__Contract,ordinal__MultipleLines,ordinal__InternetService,ordinal__OnlineSecurity,ordinal__OnlineBackup,ordinal__DeviceProtection,ordinal__TechSupport,...,remainder__OrigPDeviceProtection,remainder__OrigPTechSupport,remainder__OrigPStreamingTV,remainder__OrigPStreamingMovies,remainder__OrigPContract,remainder__OrigPPaperlessBilling,remainder__OrigPPaymentMethod,remainder__OrigPtenure,remainder__OrigPMonthlyCharges,remainder__OrigPTotalCharges
0,0.0,0.0,1.0,1.0,1.0,1.0,2.0,1.0,2.0,2.0,...,0.225021,0.151663,0.335231,0.336804,0.112695,0.335651,0.191067,0.208333,1.000000,0.000000
1,1.0,0.0,0.0,0.0,1.0,1.0,2.0,2.0,1.0,2.0,...,0.391276,0.151663,0.300702,0.336804,0.028319,0.163301,0.152431,0.164179,0.272727,0.000000
2,0.0,1.0,0.0,2.0,2.0,2.0,1.0,2.0,1.0,1.0,...,0.391276,0.416355,0.300702,0.299414,0.427097,0.335651,0.452854,0.164179,0.000000,0.000000
3,0.0,1.0,0.0,2.0,1.0,2.0,1.0,1.0,1.0,1.0,...,0.391276,0.416355,0.335231,0.336804,0.427097,0.335651,0.452854,0.619902,0.300000,0.000000
4,0.0,1.0,0.0,2.0,1.0,2.0,1.0,1.0,1.0,1.0,...,0.391276,0.416355,0.335231,0.336804,0.427097,0.335651,0.452854,0.619902,0.500000,0.666667
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
594189,0.0,0.0,0.0,0.0,2.0,2.0,1.0,1.0,2.0,1.0,...,0.225021,0.416355,0.300702,0.299414,0.028319,0.163301,0.167098,0.123077,0.000000,0.500000
594190,0.0,0.0,0.0,0.0,2.0,1.0,2.0,2.0,2.0,2.0,...,0.225021,0.151663,0.300702,0.299414,0.028319,0.163301,0.167098,0.016575,0.000000,0.000000
594191,1.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,...,0.074050,0.074050,0.074050,0.074050,0.028319,0.163301,0.152431,0.016575,0.090909,0.500000
594192,0.0,1.0,0.0,2.0,2.0,2.0,1.0,1.0,1.0,1.0,...,0.391276,0.416355,0.335231,0.299414,0.427097,0.335651,0.452854,0.275362,0.750000,0.000000


In [25]:
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(
    n_splits=10,
    shuffle=True,
    random_state=42
)

oof_preds = np.zeros(len(X_train_encoded))
test_preds = np.zeros(len(X_test_encoded))

for fold, (train_idx, valid_idx) in enumerate(skf.split(X_train_encoded, y)):

    print(f'Fold {fold+1}')

    X_train, X_valid = X_train_encoded.iloc[train_idx], X_train_encoded.iloc[valid_idx]
    y_train, y_valid = y[train_idx], y[valid_idx]

    model = XGBClassifier(
        n_estimators=10000,
        eval_metric='auc',
        random_state=42,
        early_stopping_rounds=200,
        **params
    )

    model.fit(
        X_train, y_train,
        eval_set=[(X_valid, y_valid)],
        verbose=200
    )

    oof_preds[valid_idx] = model.predict_proba(X_valid)[:, 1]
    test_preds += model.predict_proba(X_test_encoded)[:, 1] / skf.n_splits

Fold 1
[0]	validation_0-auc:0.89120
[200]	validation_0-auc:0.91265
[400]	validation_0-auc:0.91453
[600]	validation_0-auc:0.91534
[800]	validation_0-auc:0.91582
[1000]	validation_0-auc:0.91624
[1200]	validation_0-auc:0.91654
[1400]	validation_0-auc:0.91678
[1600]	validation_0-auc:0.91693
[1800]	validation_0-auc:0.91704
[2000]	validation_0-auc:0.91715
[2200]	validation_0-auc:0.91724
[2400]	validation_0-auc:0.91731
[2600]	validation_0-auc:0.91737
[2800]	validation_0-auc:0.91742
[3000]	validation_0-auc:0.91744
[3200]	validation_0-auc:0.91746
[3400]	validation_0-auc:0.91747
[3600]	validation_0-auc:0.91749
[3800]	validation_0-auc:0.91751
[4000]	validation_0-auc:0.91752
[4200]	validation_0-auc:0.91753
[4400]	validation_0-auc:0.91754
[4524]	validation_0-auc:0.91754
Fold 2
[0]	validation_0-auc:0.89014
[200]	validation_0-auc:0.91230
[400]	validation_0-auc:0.91428
[600]	validation_0-auc:0.91501
[800]	validation_0-auc:0.91547
[1000]	validation_0-auc:0.91580
[1200]	validation_0-auc:0.91605
[1400]	v

In [26]:
from sklearn.metrics import roc_auc_score

cv_score = roc_auc_score(y, oof_preds)
print("CV ROC-AUC:", cv_score)

CV ROC-AUC: 0.9175020590719831


In [27]:
submission = pd.read_csv('sample_submission.csv')
submission['Churn'] = test_preds
submission.to_csv('submission.csv', index=False)

In [28]:
with open('lightgbm-params.json', 'r') as file:
    params = json.load(file)

In [34]:
import lightgbm as lgb
from lightgbm import LGBMClassifier
from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(
    n_splits=10,
    shuffle=True,
    random_state=42
)

oof_preds = np.zeros(len(X_train_encoded))
test_preds = np.zeros(len(X_test_encoded))

for fold, (train_idx, valid_idx) in enumerate(skf.split(X_train_encoded, y)):

    print(f'Fold {fold+1}')

    X_train, X_valid = X_train_encoded.iloc[train_idx], X_train_encoded.iloc[valid_idx]
    y_train, y_valid = y[train_idx], y[valid_idx]

    model = LGBMClassifier(
        objective="binary",
        metric="auc",
        n_estimators=10000,
        verbosity=-1,
        **params
    )

    model.fit(
        X_train, y_train,
        eval_set=[(X_valid, y_valid)],
        callbacks=[lgb.log_evaluation(100),
        lgb.early_stopping(200)] 
    )

    oof_preds[valid_idx] = model.predict_proba(X_valid)[:, 1]
    test_preds += model.predict_proba(X_test_encoded)[:, 1] / skf.n_splits

Fold 1
Training until validation scores don't improve for 200 rounds
[100]	valid_0's auc: 0.914726
[200]	valid_0's auc: 0.91606
[300]	valid_0's auc: 0.916623
[400]	valid_0's auc: 0.916907
[500]	valid_0's auc: 0.917092
[600]	valid_0's auc: 0.917271
[700]	valid_0's auc: 0.917339
[800]	valid_0's auc: 0.917412
[900]	valid_0's auc: 0.917396
[1000]	valid_0's auc: 0.917404
[1100]	valid_0's auc: 0.9174
Early stopping, best iteration is:
[947]	valid_0's auc: 0.917427
Fold 2
Training until validation scores don't improve for 200 rounds
[100]	valid_0's auc: 0.914551
[200]	valid_0's auc: 0.915672
[300]	valid_0's auc: 0.916058
[400]	valid_0's auc: 0.916342
[500]	valid_0's auc: 0.916491
[600]	valid_0's auc: 0.916578
[700]	valid_0's auc: 0.916626
[800]	valid_0's auc: 0.916718
[900]	valid_0's auc: 0.916773
[1000]	valid_0's auc: 0.916764
[1100]	valid_0's auc: 0.916794
[1200]	valid_0's auc: 0.916799
[1300]	valid_0's auc: 0.91682
[1400]	valid_0's auc: 0.916798
Early stopping, best iteration is:
[1247]	va

In [35]:
cv_score = roc_auc_score(y, oof_preds)
print("CV ROC-AUC:", cv_score)

CV ROC-AUC: 0.917371045895405


In [36]:
submission = pd.read_csv('sample_submission.csv')
submission['Churn'] = test_preds
submission.to_csv('submission.csv', index=False)